# Stage 4 – Validation and data-quality report

**Goal:** prove that the partnership table is correct before we build any network.

We run the structural checks, the independent scorecard checks and the spot checks from
`library/lib_03_validation.ipynb`, and write everything to `docs/data_quality_report.md`.

**Gate:** if any check fails, or if the extractor raised a warning, this notebook stops with an error.

**Reads:** the tables from Stages 2 and 3 and the raw match files
**Writes:** `docs/data_quality_report.md`, `outputs/tables/validation_*.csv`, `outputs/tables/dataset_counts.csv`

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_01_data_loading.ipynb
%run ./library/lib_03_validation.ipynb
stage_log = get_stage_logger("04_validate")

In [2]:
included_matches = read_table(PROCESSED_FOLDER / "match_metadata.csv")
observations = read_table(PROCESSED_FOLDER / "partnership_observations.csv")
innings_table = read_table(PROCESSED_FOLDER / "innings_summary.csv")
extraction_warnings = read_table(PROCESSED_FOLDER / "extraction_warnings.csv")
file_audit = read_table(INTERIM_FOLDER / "file_audit.csv")

### 1. Run the three kinds of checks

In [3]:
structural_results = pd.DataFrame(observation_table_checks(observations, innings_table),
                                  columns=["check", "passed", "detail"])
scorecard_results = pd.DataFrame([
    check_row
    for match_record in included_matches.to_dict("records")
    for check_row in scorecard_checks(load_match_file(RAW_MATCH_FOLDER / match_record["filename"]),
                                      match_record["match_id"], match_record["format"])])
spot_check_results = documented_partnership_checks(observations)

write_table(structural_results, TABLES_FOLDER / "validation_structural_checks.csv")
write_table(scorecard_results, TABLES_FOLDER / "validation_scorecard_checks.csv")
write_table(spot_check_results, TABLES_FOLDER / "validation_spot_checks.csv")
spot_check_results

,format,match_date,pair,documented_runs,extracted_runs,passed
0,Test,2020-08-21,Crawley / Buttler,359,359,True
1,Test,2022-07-01,Root / Bairstow,269,269,True
2,Test,2022-12-01,Crawley / Duckett,233,233,True
3,Test,2024-10-07,Root / Brook,454,454,True
4,T20I,2022-11-10,Buttler / Hales,170,170,True


### 2. Counts per format

In [4]:
qualifying_innings = innings_table[innings_table["qualifying"]]
dataset_counts = pd.DataFrame({
    "matches": included_matches.groupby("format").size(),
    "matches_with_england_innings": qualifying_innings.groupby("format")["match_id"].nunique(),
    "england_innings": qualifying_innings.groupby("format").size(),
    "observations": observations.groupby("format").size(),
    "partnership_runs": observations.groupby("format")["partnership_runs"].sum(),
    "wicket_records": qualifying_innings.groupby("format")["wicket_records"].sum(),
    "deliveries": qualifying_innings.groupby("format")["delivery_count"].sum(),
}).reindex(list(FORMATS)).reset_index()
write_table(dataset_counts, TABLES_FOLDER / "dataset_counts.csv")
dataset_counts

,format,matches,matches_with_england_innings,england_innings,observations,partnership_runs,wicket_records,deliveries
0,Test,81,81,151,1357,43080,1328,66829
1,ODI,88,85,85,708,22929,662,22766
2,T20I,118,114,114,763,18974,675,12878


### 3. Write the data-quality report

In [5]:
scorecard_summary = (scorecard_results.groupby(["format", "check"])["passed"]
                     .agg(checked="size", passed="sum").reset_index())
failed_scorecard_checks = scorecard_results[~scorecard_results["passed"]]
special_innings = innings_table[(innings_table["penalty_runs_pre"] + innings_table["penalty_runs_post"] > 0)
                                | innings_table["absent_hurt"].notna() | innings_table["super_over"]]
unusual_endings = observations[observations["boundary_reason"].str.startswith(("replacement", "retirement", "unexplained"))]
matches_without_england_innings = included_matches[included_matches["n_innings_england"] == 0]
unreadable_files = int(file_audit["exclusion_reason"].astype(str).str.startswith("unreadable").sum())

report_lines = [
    "# Data-quality report", "",
    "Generated by `notebooks/04_validate_observations.ipynb`. Do not edit by hand.", "",
    "## Source files", "",
    f"- JSON files in archive: {len(file_audit)}; unreadable: {unreadable_files}",
    f"- Included matches: {int(file_audit['included'].sum())}; excluded: {int((~file_audit['included']).sum())}",
    "", "Exclusion reasons (one per excluded file):", "",
    markdown_table(file_audit.loc[~file_audit["included"], "exclusion_reason"].value_counts()
                   .rename_axis("reason").reset_index(name="files")),
    "", "## Counts by format", "", markdown_table(dataset_counts), "",
    f"Included matches in which England never batted (abandoned / no result before England's innings): "
    f"{len(matches_without_england_innings)} ({', '.join(matches_without_england_innings['match_id'].astype(str))}). "
    "They stay in the match table but yield no observations.",
    "", "## Structural checks", "", markdown_table(structural_results), "",
    "## Independent scorecard checks", "",
    "Innings totals recomputed from raw deliveries (without the extractor) compared with the target and "
    "run margin recorded by Cricsheet. Matches decided with a rain-adjustment method are not checked.", "",
    markdown_table(scorecard_summary), "",
    f"Failed scorecard checks: {len(failed_scorecard_checks)}", "",
    markdown_table(failed_scorecard_checks) if len(failed_scorecard_checks) else "", "",
    "## Spot checks against documented partnerships", "", markdown_table(spot_check_results), "",
    "## Extraction warnings", "",
    markdown_table(extraction_warnings) if len(extraction_warnings) else
    "None: every pair change is explained by a wicket record, a batter replacement record or the end of the innings.", "",
    "## Partnership boundaries other than dismissals / innings end", "",
    markdown_table(unusual_endings[["match_id", "format", "match_date", "innings_index", "player_1_name",
                                    "player_2_name", "partnership_runs", "boundary_reason"]]), "",
    "## Innings with penalty runs, absent batters or super overs", "",
    markdown_table(special_innings[["match_id", "format", "innings_index", "penalty_runs_pre", "penalty_runs_post",
                                    "absent_hurt", "super_over", "runs_from_deliveries"]].fillna("")), "",
    "Innings-level penalty runs are not part of any delivery and are therefore not assigned to any pair.", "",
]
(PROJECT_ROOT / "docs" / "data_quality_report.md").write_text("\n".join(report_lines), encoding="utf-8")

3600

### 4. Gate

In [6]:
all_checks_passed = bool(structural_results["passed"].all() and scorecard_results["passed"].all()
                         and spot_check_results["passed"].all() and len(extraction_warnings) == 0)
stage_log.info("structural checks passed: %d/%d", structural_results["passed"].sum(), len(structural_results))
stage_log.info("scorecard checks passed: %d/%d", scorecard_results["passed"].sum(), len(scorecard_results))
stage_log.info("spot checks passed: %d/%d", spot_check_results["passed"].sum(), len(spot_check_results))
stage_log.info("extraction warnings: %d", len(extraction_warnings))
stage_log.info("GATE %s", "PASSED" if all_checks_passed else "FAILED - see docs/data_quality_report.md")
assert all_checks_passed, "Validation gate failed - see docs/data_quality_report.md"

2026-11-16 20:50:11,382 INFO batnet.04_validate: structural checks passed: 13/13


2026-11-16 20:50:11,383 INFO batnet.04_validate: scorecard checks passed: 318/318


2026-11-16 20:50:11,383 INFO batnet.04_validate: spot checks passed: 5/5


2026-11-16 20:50:11,383 INFO batnet.04_validate: extraction warnings: 0


2026-11-16 20:50:11,384 INFO batnet.04_validate: GATE PASSED
